In [94]:
import os
from pathlib import Path

import pandas as pd
import polars as pl
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import scipy
import sklearn
import tensorflow as tf
import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import mllabs  # ml-labs: our experiment management framework

import sys
print(sys.version)

for i in [pd, pl, np, plt, sns, scipy, sklearn, tf, xgb, lgb, cb, mllabs]:
    if hasattr(i, '__version__'):
        print(i.__name__, i.__version__)
    else:
        print(i.__name__)

3.12.12 (main, Dec 27 2025, 11:08:36) [GCC 13.3.0]
pandas 2.3.3
polars 1.38.1
numpy 2.3.5
matplotlib.pyplot
seaborn 0.13.2
scipy 1.16.3
sklearn 1.8.0
tensorflow 2.20.0
xgboost 3.2.0
lightgbm
catboost 1.2.10
mllabs 0.6.3


In [95]:
from IPython.display import Markdown

from mllabs import Connector, Experimenter, ColSelector
from mllabs.collector import MetricCollector, ModelAttrCollector
from mllabs.adapter import XGBoostAdapter, LightGBMAdapter, CatBoostAdapter
from mllabs.nn import NNClassifier
from mllabs.col import ohe_drop_first

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [3]:
from mllabs.processor import PolarsLoader, PandasConverter, ExprProcessor
from sklearn.pipeline import make_pipeline

data_path = Path('data')

# Binary-encode several categorical features using Polars expressions.
# We also derive indicator columns for internet-service-related features
# ('No internet service' is treated as a separate binary flag).
dict_expr = {
    'gender': (pl.col('gender') == 'Male').cast(pl.Int8),
    'No_Internet': (pl.col('DeviceProtection') == 'No internet service').cast(pl.Int8),
    'DSL_Y': (pl.col('InternetService') == 'DSL').cast(pl.Int8)
}
for i in ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService']:
    dict_expr[i] = (pl.col(i) == 'Yes').cast(pl.Int8)

for i in ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport', 'MultipleLines']:
    dict_expr[i + '_Y'] = (pl.col(i) == 'Yes').cast(pl.Int8)

# PolarsLoader reads CSVs into Polars DataFrames.
# ExprProcessor applies the expression dict above in a single vectorized pass.
loader = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor(dict_expr=dict_expr)
)

df_train = loader.fit_transform([data_path / 'train.csv']).with_columns(
    (pl.col('Churn') == 'Yes').cast(pl.Int8)
)
df_org = loader.transform([data_path / 'WA_Fn-UseC_-Telco-Customer-Churn.csv']).drop('customerID').with_columns(
    id=-pl.int_range(1, pl.len() + 1),
    tenure=pl.col('tenure').clip(1),
    Churn=(pl.col('Churn') == 'Yes').cast(pl.Int8)
)
df_test = loader.transform([data_path / 'test.csv'])

In [4]:
# Variable groupings
# X_bin  : original binary features (Yes/No encoded as 0/1)
# X_tri  : 3-level categorical features (Yes / No / No internet service)
# X_bin2 : derived binary flags from X_tri (e.g. OnlineSecurity_Y)
# X_num  : numerical features
# X_nom  : nominal categoricals kept as strings for tree-based models
X_bin = ['Dependents', 'PaperlessBilling', 'Partner', 'PhoneService', 'gender', 'SeniorCitizen']
X_tri = ['DeviceProtection', 'OnlineBackup', 'OnlineSecurity', 'StreamingMovies', 'StreamingTV', 'TechSupport']
X_bin2 = ['{}_Y'.format(i) for i in X_tri] + ['No_Internet', 'DSL_Y', 'MultipleLines_Y']
X_tri.append('InternetService')
X_tri.append('MultipleLines')
X_num = ['TotalCharges', 'MonthlyCharges', 'tenure']
X_nom = ['PaymentMethod', 'Contract']
X_all = X_bin + X_tri + X_bin2 + X_num + X_nom

target = 'Churn'

In [5]:
if os.path.exists('exp/ss1_aug'):
    e_model = Experimenter.load('exp/ss1_aug', df_train, aug_data=df_org)
    if e_model.status == 'closed':
        e_model.reopen_exp()
else:
    e_model = Experimenter.create(
        df_train, 'exp/ss1_aug', title='The experimentation for making ML models',
        sp=StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=1),
        sp_v=StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=1),
        splitter_params={'y': target}, aug_data=df_org
    )
Markdown(
    e_model.desc_spec()
)

Loaded: 35 node(s), 8 group(s), 1 fold(s)


## The experimentation for making ML models

| 항목 | 값 |
|------|-----|
| **Outer Splitter (sp)** | `StratifiedShuffleSplit(n_splits=1, random_state=1, test_size=0.2)` |
| **Inner Splitter (sp_v)** | `StratifiedShuffleSplit(n_splits=1, random_state=1, test_size=0.1)` |
| **Splitter Params** | `{y='Churn'}` |
| **Outer Folds** | 1 |
| **Inner Folds** | 1 |

In [6]:
# Pipeline setup
# ml-labs uses a node graph: Stage nodes (preprocessing) → Head nodes (models).
# set_grp defines a group template; set_node creates individual experiment nodes.

e_model.set_grp('pre', role='stage', method='transform')
y_edges = {'y': [(None, target)]}
e_model.set_grp(
    'clf', role='head', method='predict_proba', edges=y_edges
)

# XGBoost — uses XGBoostAdapter to pass eval_set and early_stopping_rounds
e_model.set_grp('xgb', parent='clf', processor=xgb.XGBClassifier,
    adapter=XGBoostAdapter(eval_mode='valid'),
    params={
        'n_estimators': 10000, 'learning_rate': 0.05, 'early_stopping_rounds': 50,
        'eval_metric': 'auc', 'enable_categorical': True, 'device': 'cuda',
        'verbosity': 0, 'random_state': 1,
    }
)
e_model.add_collector(
    ModelAttrCollector('xgb_evals_results', Connector(processor=xgb.XGBClassifier), 'evals_result')
)

# LightGBM — early_stopping passed as dict; LightGBMAdapter creates the callback internally
e_model.set_grp('lgb', parent='clf', processor=lgb.LGBMClassifier,
    adapter=LightGBMAdapter(eval_mode='valid'),
    params={
        'n_estimators': 10000, 'learning_rate': 0.05,
        'early_stopping': {'stopping_rounds': 50, 'first_metric_only': True},
        'eval_metric': 'auc', 'verbose': -1, 'random_state': 1,
    }
)
e_model.add_collector(
    ModelAttrCollector('lgb_evals_results', Connector(processor=lgb.LGBMClassifier), 'evals_result')
)

# CatBoost — CatBoostAdapter handles cat_features and eval_set
e_model.set_grp('cb', parent='clf', processor=cb.CatBoostClassifier,
    adapter=CatBoostAdapter(eval_mode='valid'),
    params={
        'iterations': 10000, 'learning_rate': 0.05, 'early_stopping_rounds': 50,
        'eval_metric': 'AUC', 'verbose': 0, 'random_state': 1, 'task_type': 'GPU',
    }
)
e_model.add_collector(
    ModelAttrCollector('cb_evals_results', Connector('_base$', processor=cb.CatBoostClassifier), 'evals_result')
)

# LogisticRegression
e_model.set_grp('lr', parent='clf', processor=LogisticRegression,
    params={
        'max_iter': 1000, 'random_state': 1,
    }
)

## Neural network
e_model.set_grp('nn', parent = 'clf', processor = NNClassifier, params = {'metrics': ['auc'], 'early_stopping': 10})
e_model.add_collector(
    ModelAttrCollector('nn_evals', Connector(processor=NNClassifier), result_key='evals_result')
)

e_model.add_collector(
    MetricCollector(
        'AUC',
        Connector(edges=y_edges), slice(-1, None), roc_auc_score, include_train = True
    )
)

In [129]:
e_model.set_grp(
    'xgb_max_depth3_base', parent='xgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
    params = {'max_depth': 3, 'enable_categorical': True, 'device': 'cuda',
        'verbosity': 0, 'random_state': 1,
        'learning_rate': 0.05, 'colsample_bytree': 0.75, 'subsample': 0.9, 'reg_lambda': 0.01}
)
e_model.set_node('xgb_max_depth3_base_1', grp='xgb_max_depth3_base')
e_model.exp()

Experimenting 2 node(s)
Exp 0/1 (0%) > lgb_base9 0/2 (0%) > 1/10000 (0%) valid_0-auc: 0.8758, valid_0-binary_logloss: 0.5158Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_base9 0/2 (0%) > 1000/10000 (10%) valid_0-auc: 0.9181, valid_0-binary_logloss: 0.2951Early stopping, best iteration is:
[1778]	valid_0's auc: 0.918486	valid_0's binary_logloss: 0.294475
Evaluated only: auc
Exp 1/1 (100%)                                                                                                   
Experimentation complete: 2 node(s)


In [13]:
e_model.set_node(
    'lgb_base', grp='lgb', edges={'X': [(None, X_bin + X_nom + X_bin2 + X_num)]}, 
    params = {'enable_categorical': True,
        'num_leaves': 7, 'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)
e_model.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb_base 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.8725, valid_0-binary_logloss: 0.5163Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_base 0/1 (0%) > 2000/10000 (20%) valid_0-auc: 0.9163, valid_0-binary_logloss: 0.2979Early stopping, best iteration is:
[1995]	valid_0's auc: 0.916346	valid_0's binary_logloss: 0.297872
Evaluated only: auc
Exp 1/1 (100%)                                                                                         
Experimentation complete: 1 node(s)


In [14]:
e_model.set_node(
    'lgb_base2', grp='lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num)]}, 
    params = {'enable_categorical': True,
        'num_leaves': 7, 'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)
e_model.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb_base2 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.8724, valid_0-binary_logloss: 0.5165Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_base2 0/1 (0%) > 1000/10000 (10%) valid_0-auc: 0.9158, valid_0-binary_logloss: 0.2987Early stopping, best iteration is:
[1805]	valid_0's auc: 0.916331	valid_0's binary_logloss: 0.297886
Evaluated only: auc
Exp 1/1 (100%)                                                                                          
Experimentation complete: 1 node(s)


In [15]:
e_model.get_collector('AUC').get_metrics_agg()[0]

,valid,train_sub,valid_sub
lgb_base,0.915662,0.917639,0.916346
lgb_base2,0.915639,0.917468,0.916331
xgb_max_depth3_base_1,0.915819,0.917992,0.916477


In [81]:
from sklearn.preprocessing import KBinsDiscretizer, RobustScaler, PolynomialFeatures, StandardScaler, TargetEncoder, MinMaxScaler
from mllabs.processor import FrequencyEncoder, CatConverter, CatOOVFilter, TypeConverter
from sklearn.impute import SimpleImputer
e_model.set_node(
    'si', grp='pre', processor=SimpleImputer, edges = {'X': [(None, X_num)]}
)
e_model.set_node(
    'kbin', grp='pre', processor=KBinsDiscretizer, edges = {'X': [('si', None)]}, params = {'n_bins': 10, 'strategy': 'uniform', 'encode': 'ordinal', 'random_state': 123}
)
e_model.set_node(
    'rb', grp='pre', processor=RobustScaler, edges = {'X': [('si', None)]}, params = {}
)
e_model.set_node(
    'std', grp='pre', processor=StandardScaler, edges = {'X': [('si', None)]}, params = {}
)
e_model.set_node(
    'pf2', grp='pre', processor=PolynomialFeatures, edges = {'X': [('std', None)]}, params = {'degree': 2, 'interaction_only': True, 'include_bias': False}
)
e_model.set_node(
    'pf2_not_std', grp='pre', processor=PolynomialFeatures, edges = {'X': [('si', None)]}, params = {'degree': 2, 'interaction_only': True, 'include_bias': False}
)
e_model.set_node(
    'tgt_num', grp='pre', processor=TargetEncoder, method = 'fit_transform', edges = {'X': [('si', None)], **y_edges}, params = {'target_type': 'binary'}
)
e_model.set_node(
    'freq_num', grp='pre', processor=FrequencyEncoder, method = 'transform', edges = {'X': [('si', None)]}, params = {'normalize': True}
)

e_model.set_node(
    'n2s', grp='pre', processor=TypeConverter, method = 'transform', edges = {'X': [('si', None)]}, params={'to': 'str'}
)

e_model.set_node(
    'n2c', grp='pre', processor=CatConverter, method = 'transform', edges = {'X': [('n2s', None), ('kbin', None)]}
)

e_model.set_node(
    'coov', grp='pre', processor=CatOOVFilter, method = 'transform', edges = {'X': [('n2c', None)]}, params={'min_frequency': 10, 'missing_value': 'OOV'}
)

e_model.set_node(
    'rb_std', grp='pre', processor=RobustScaler, edges = {'X': [('rb', None)]}, params = {}
)
e_model.set_node(
    'mm', grp='pre', processor=MinMaxScaler, edges = {'X': [(None, X_num)]}, params = {}
)
e_model.set_node(
    'expr3', grp='pre', processor = ExprProcessor, edges = {'X': [('si', None)]},
    params={'dict_expr': {
        'TotalCharges_per_tenure': pl.col('si__TotalCharges') / pl.col('si__tenure'),
    }, 'with_columns': False}
)
e_model.set_node(
    'expr4', grp='pre', processor = ExprProcessor, edges = {'X': [('si', None)]},
    params={'dict_expr': {
        'TotalCharges_per_tenure': pl.col('si__TotalCharges') / pl.col('si__tenure'),
        'Total_per_Monthly_ratio': pl.col('si__TotalCharges') / pl.col('si__MonthlyCharges'),
        'TotalCharges_per_tenure_min': pl.min_horizontal(pl.col('si__TotalCharges') / pl.col('si__tenure'), pl.col('si__MonthlyCharges')),
        'TotalCharges_per_tenure_max': pl.max_horizontal(pl.col('si__TotalCharges') / pl.col('si__tenure'), pl.col('si__MonthlyCharges')),
    }, 'with_columns': False}
)
service = ['DeviceProtection_Y', 'OnlineBackup_Y', 'OnlineSecurity_Y', 'StreamingMovies_Y', 'StreamingTV_Y', 'TechSupport_Y']
e_model.set_node(
    'expr5', grp='pre', processor = ExprProcessor, edges = {'X': [(None, service)]},
    params={'dict_expr': {
        'Service_cnt':  pl.sum_horizontal(service),
    }, 'with_columns': False}
)
e_model.set_node(
    'expr3_std', grp='pre', processor = StandardScaler, edges = {'X': [('expr3', None)]}
)

e_model.build()

Affected 4 dependent node(s): ['lgb_base10', 'lgb_base11', 'lgb_base12', 'lgb_base13']
Building 1 node(s)
Build 1/1 (100%)                 
Build complete: 1 node(s)


In [60]:
from mllabs.processor import CatPairCombiner
from itertools import combinations

TOP_CATS_FOR_NGRAM = ['Contract', 'DSL_Y', 'PaymentMethod', 'OnlineSecurity']

bigram_cols = [(i1, i2) for i1, i2 in combinations(TOP_CATS_FOR_NGRAM, 2)]
trigram_cols = [(i1, i2, i3) for i1, i2, i3 in combinations(TOP_CATS_FOR_NGRAM, 3)]

e_model.set_node(
    'bigram', grp='pre', processor = CatPairCombiner, edges = {'X': [(None, TOP_CATS_FOR_NGRAM)]}, params={'pairs': bigram_cols}, exist='replace'
)
e_model.set_node(
    'trigram', grp='pre', processor = CatPairCombiner, edges = {'X': [(None, TOP_CATS_FOR_NGRAM[:4])]}, params={'pairs': trigram_cols}, exist='replace'
)
e_model.set_node(
    'tgt_bigram', grp='pre', processor=TargetEncoder, method = 'fit_transform', edges = {'X': [('bigram', None)], **y_edges}, params = {'target_type': 'binary'}
)
e_model.set_node(
    'tgt_trigram', grp='pre', processor=TargetEncoder, method = 'fit_transform', edges = {'X': [('trigram', None)], **y_edges}, params = {'target_type': 'binary'}
)
e_model.build()

Affected 3 dependent node(s): ['lgb_base11', 'lgb_base12', 'tgt_bigram']
Affected 2 dependent node(s): ['lgb_base12', 'tgt_trigram']
Building 4 node(s)
Build 1/1 (100%)                       
Build complete: 4 node(s)


In [20]:
e_model.set_node(
    'lgb_base23', grp='lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num), ('kbin', None)]}, 
    params = {'enable_categorical': True,
        'num_leaves': 7, 'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)
e_model.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb_base23 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.8737, valid_0-binary_logloss: 0.5152Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_base23 0/1 (0%) > 2000/10000 (20%) valid_0-auc: 0.9164, valid_0-binary_logloss: 0.2978Early stopping, best iteration is:
[2797]	valid_0's auc: 0.916564	valid_0's binary_logloss: 0.29753
Evaluated only: auc
Exp 1/1 (100%)                                                                                           
Experimentation complete: 1 node(s)


In [21]:
e_model.get_collector('AUC').get_metrics_agg()[0]

,valid,train_sub,valid_sub
lgb_base,0.915662,0.917639,0.916346
lgb_base2,0.915639,0.917468,0.916331
lgb_base23,0.915847,0.918547,0.916564
xgb_max_depth3_base_1,0.915819,0.917992,0.916477


In [22]:
e_model.set_node(
    'lgb_base4', grp='lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num), ('kbin', None), ('rb', None)]}, 
    params = {'enable_categorical': True,
        'num_leaves': 7, 'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)
e_model.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb_base4 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.8761, valid_0-binary_logloss: 0.5157Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_base4 0/1 (0%) > 2000/10000 (20%) valid_0-auc: 0.9166, valid_0-binary_logloss: 0.2975Early stopping, best iteration is:
[2231]	valid_0's auc: 0.916678	valid_0's binary_logloss: 0.297359
Evaluated only: auc
Exp 1/1 (100%)                                                                                          
Experimentation complete: 1 node(s)


In [23]:
e_model.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending=False)

,valid,train_sub,valid_sub
lgb_base4,0.915891,0.918233,0.916678
lgb_base23,0.915847,0.918547,0.916564
xgb_max_depth3_base_1,0.915819,0.917992,0.916477
lgb_base,0.915662,0.917639,0.916346
lgb_base2,0.915639,0.917468,0.916331


In [24]:
e_model.set_node(
    'lgb_base5', grp='lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num), ('kbin', None), ('rb', None), ('pf2', None)]}, 
    params = {'enable_categorical': True,
        'num_leaves': 7, 'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)
e_model.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb_base5 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.8703, valid_0-binary_logloss: 0.5156Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_base5 0/1 (0%) > 1000/10000 (10%) valid_0-auc: 0.9160, valid_0-binary_logloss: 0.2984Early stopping, best iteration is:
[1858]	valid_0's auc: 0.916608	valid_0's binary_logloss: 0.297489
Evaluated only: auc
Exp 1/1 (100%)                                                                                          
Experimentation complete: 1 node(s)


In [25]:
e_model.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending=False)

,valid,train_sub,valid_sub
lgb_base4,0.915891,0.918233,0.916678
lgb_base23,0.915847,0.918547,0.916564
xgb_max_depth3_base_1,0.915819,0.917992,0.916477
lgb_base5,0.915736,0.918005,0.916608
lgb_base,0.915662,0.917639,0.916346
lgb_base2,0.915639,0.917468,0.916331


In [27]:
e_model.set_node(
    'lgb_base6', grp='lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num), ('kbin', None), ('rb', None), ('pf2_not_std', None)]}, 
    params = {'enable_categorical': True,
        'num_leaves': 7, 'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)
e_model.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb_base6 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.8750, valid_0-binary_logloss: 0.5159Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_base6 0/1 (0%) > 2000/10000 (20%) valid_0-auc: 0.9165, valid_0-binary_logloss: 0.2976Early stopping, best iteration is:
[2053]	valid_0's auc: 0.91653	valid_0's binary_logloss: 0.297576
Evaluated only: auc
Exp 1/1 (100%)                                                                                          
Experimentation complete: 1 node(s)


In [28]:
e_model.set_node(
    'lgb_base7', grp='lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num), ('kbin', None), ('rb', None), ('tgt_num', None)]}, 
    params = {'enable_categorical': True,
        'num_leaves': 7, 'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)
e_model.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb_base7 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.8745, valid_0-binary_logloss: 0.5160Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_base7 0/1 (0%) > 1000/10000 (10%) valid_0-auc: 0.9177, valid_0-binary_logloss: 0.2957Early stopping, best iteration is:
[1668]	valid_0's auc: 0.918097	valid_0's binary_logloss: 0.295127
Evaluated only: auc
Exp 1/1 (100%)                                                                                          
Experimentation complete: 1 node(s)


In [29]:
e_model.set_node(
    'lgb_base8', grp='lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None)]}, 
    params = {'enable_categorical': True,
        'num_leaves': 7, 'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)
e_model.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb_base8 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.8746, valid_0-binary_logloss: 0.5160Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_base8 0/1 (0%) > 2000/10000 (20%) valid_0-auc: 0.9183, valid_0-binary_logloss: 0.2948Early stopping, best iteration is:
[2037]	valid_0's auc: 0.918273	valid_0's binary_logloss: 0.294823
Evaluated only: auc
Exp 1/1 (100%)                                                                                          
Experimentation complete: 1 node(s)


In [30]:
e_model.set_node(
    'lgb_base9', grp='lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr3', None)]}, 
    params = {'enable_categorical': True,
        'num_leaves': 7, 'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)
e_model.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb_base9 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.8758, valid_0-binary_logloss: 0.5158Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_base9 0/1 (0%) > 1000/10000 (10%) valid_0-auc: 0.9181, valid_0-binary_logloss: 0.2951Early stopping, best iteration is:
[1833]	valid_0's auc: 0.918477	valid_0's binary_logloss: 0.294498
Evaluated only: auc
Exp 1/1 (100%)                                                                                          
Experimentation complete: 1 node(s)


In [139]:
e_model.set_node(
    'lgb_base10', grp='lgb', edges={'X': [(None, X_bin + X_nom + X_tri + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None)]}, 
    params = {'enable_categorical': True,
        'num_leaves': 7, 'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)
e_model.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb_base10 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.8674, valid_0-binary_logloss: 0.5165Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_base10 0/1 (0%) > 1000/10000 (10%) valid_0-auc: 0.9180, valid_0-binary_logloss: 0.2953Early stopping, best iteration is:
[1647]	valid_0's auc: 0.918367	valid_0's binary_logloss: 0.294669
Evaluated only: auc
Exp 1/1 (100%)                                                                                           
Experimentation complete: 1 node(s)


In [64]:
e_model.set_node(
    'lgb_base11', grp='lgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('tgt_bigram', None)]
    }, params = {'enable_categorical': True,
        'num_leaves': 7, 'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)
e_model.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb_base11 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.8716, valid_0-binary_logloss: 0.5154Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_base11 0/1 (0%) > 1000/10000 (10%) valid_0-auc: 0.9181, valid_0-binary_logloss: 0.2952Early stopping, best iteration is:
[1899]	valid_0's auc: 0.91849	valid_0's binary_logloss: 0.294494
Evaluated only: auc
Exp 1/1 (100%)                                                                                           
Experimentation complete: 1 node(s)


In [86]:
e_model.set_node(
    'lgb_base12', grp='lgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('tgt_bigram', None), ('tgt_trigram', None)]
    }, params = {'enable_categorical': True,
        'num_leaves': 7, 'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)
e_model.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb_base12 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.8833, valid_0-binary_logloss: 0.5148Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_base12 0/1 (0%) > 1000/10000 (10%) valid_0-auc: 0.9180, valid_0-binary_logloss: 0.2953Early stopping, best iteration is:
[1924]	valid_0's auc: 0.91841	valid_0's binary_logloss: 0.294606
Evaluated only: auc
Exp 1/1 (100%)                                                                                           
Experimentation complete: 1 node(s)


In [92]:
e_model.set_node(
    'lgb_base13', grp='lgb', edges={
        'X': [(None, X_bin + X_nom + X_bin2 + X_num), ('kbin', None), ('rb', None), ('tgt_num', None), ('freq_num', None), ('expr4', None), ('expr5', None)]
    }, params = {'enable_categorical': True,
        'num_leaves': 7, 'learning_rate': 0.05, 'verbose': -1, 'random_state': 1, 'min_samples_leaf': 512, 'colsample_bytree': 0.75, 
        'subsample': 0.75, 'subsample_freq': 1,'reg_lambda': 1
    }
)
e_model.exp()

Experimenting 1 node(s)
Exp 0/1 (0%) > lgb_base13 0/1 (0%) > 1/10000 (0%) valid_0-auc: 0.8746, valid_0-binary_logloss: 0.5160Training until validation scores don't improve for 50 rounds
Exp 0/1 (0%) > lgb_base13 0/1 (0%) > 2000/10000 (20%) valid_0-auc: 0.9187, valid_0-binary_logloss: 0.2942Early stopping, best iteration is:
[2289]	valid_0's auc: 0.91874	valid_0's binary_logloss: 0.29408
Evaluated only: auc
Exp 1/1 (100%)                                                                                           
Experimentation complete: 1 node(s)


In [93]:
e_model.get_collector('AUC').get_metrics_agg()[0].sort_values('valid', ascending=False).iloc[:10]

,valid,train_sub,valid_sub
lgb_base13,0.917405,0.921602,0.918740
lgb_base10,0.917317,0.920754,0.918519
lgb_base12,0.917234,0.921303,0.918410
lgb_base9,0.917202,0.920530,0.918486
lgb_base11,0.917193,0.920372,0.918397
lgb_base8,0.917163,0.920650,0.918273
lgb_base7,0.917139,0.919628,0.918097
lgb_base4,0.915891,0.918233,0.916678
lgb_base23,0.915847,0.918547,0.916564
xgb_max_depth3_base_1,0.915819,0.917992,0.916477


In [38]:
e_model.set_node(
    'nn_128_64', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('coov', None), ('tgt_num', None), ('freq_num', None)]},
    params={'hidden': {'units': [128, 64]}, 'cat_cols': ColSelector(col_type = 'category')}
)
e_model.exp()

Experimenting 0 node(s)
Exp 1/1 (100%)       
Experimentation complete: 0 node(s)


In [85]:
e_model.set_node(
    'nn_256_128', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('coov', None), ('tgt_num', None), ('freq_num', None)]},
    params={'hidden': {'units': [256, 128]}, 'cat_cols': ColSelector(col_type = 'category')}
)
e_model.exp()

Experimenting 1 node(s)
Exp 1/1 (100%)                                                                                                
Experimentation complete: 1 node(s)


In [147]:
e_model.set_node(
    'nn_256_128_2', grp='nn', 
    edges={'X': [(None, X_bin + X_nom + X_bin2), ('std', None), ('coov', None), ('tgt_num', None), ('freq_num', None), ('expr3_std', None)]},
    params={'hidden': {'units': [256, 128]}, 'cat_cols': ColSelector(col_type = 'category')}
)
e_model.exp()

Experimenting 1 node(s)
Exp 1/1 (100%)                                                                                                  
Experimentation complete: 1 node(s)


In [148]:
e_model.get_collector('AUC').get_metrics_agg('nn_.*')[0].sort_values('valid', ascending=False)

,valid,train_sub,valid_sub
nn_256_128_2,0.915740,0.919513,0.916715
nn_256_128,0.915395,0.919802,0.916465
nn_128_64,0.915307,0.919519,0.916422
